## はじめに

このノートブックでは、非構造化データに対するCortex Search Serviceを作成します。

**主な処理内容:**
- FAQドキュメント用Cortex Search Serviceの作成
- 運営マニュアル用Cortex Search Serviceの作成
- 音声ログ要約用Cortex Search Serviceの作成
- SNS投稿分析用Cortex Search Serviceの作成
- 各Serviceの検索テスト

**Cortex Searchとは:**
- 非構造化データに対するハイブリッド検索（キーワード＋セマンティック）を実現
- 自動的にベクトル埋め込みを生成し、検索インデックスを構築
- Cortex AgentやMCP Serverのツールとして連携可能

---

### 作成するCortex Search Service

| サービス名 | 対象テーブル | 検索列 | 属性列 |
|-----------|-------------|--------|--------|
| SEARCH_FAQ | GOLD_FAQ_DOCUMENTS | CONTENT_CHUNK | なし |
| SEARCH_OPERATION_MANUALS | GOLD_OPERATION_MANUALS | CONTENT_CHUNK | なし |
| SEARCH_VOICE_LOGS | GOLD_VOICE_LOGS | TRANSCRIBED_TEXT_SUMMARY | CALL_ID, CATEGORY, INQUIRY_CATEGORY, SENTIMENT |
| SEARCH_SNS_MENTIONS | GOLD_SNS_MENTIONS_ANALYZED | CONTENT | PLATFORM, POST_CATEGORY, SENTIMENT |

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
USE WAREHOUSE MCP_HANDSON_WH;
USE SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

## 1. 対象データの確認

Cortex Search Serviceを作成する前に、対象となるテーブルのデータを確認します。

In [ ]:
-- ============================================================================
-- 対象テーブルのレコード数確認
-- ============================================================================
SELECT 
    'GOLD_FAQ_DOCUMENTS' AS table_name,
    COUNT(*) AS record_count,
    'FAQドキュメント' AS description
FROM GOLD_FAQ_DOCUMENTS
UNION ALL
SELECT 
    'GOLD_OPERATION_MANUALS' AS table_name,
    COUNT(*) AS record_count,
    '運営マニュアル' AS description
FROM GOLD_OPERATION_MANUALS
UNION ALL
SELECT 
    'GOLD_VOICE_LOGS' AS table_name,
    COUNT(*) AS record_count,
    '音声ログ（要約付き）' AS description
FROM GOLD_VOICE_LOGS
UNION ALL
SELECT 
    'GOLD_SNS_MENTIONS_ANALYZED' AS table_name,
    COUNT(*) AS record_count,
    'SNS投稿分析済み' AS description
FROM GOLD_SNS_MENTIONS_ANALYZED
ORDER BY table_name;

In [ ]:
-- ============================================================================
-- FAQドキュメントのサンプルデータ確認
-- ============================================================================
SELECT 
    RELATIVE_PATH,
    LEFT(CONTENT_CHUNK, 200) AS content_preview
FROM GOLD_FAQ_DOCUMENTS
LIMIT 5;

In [ ]:
-- ============================================================================
-- SNS投稿のサンプルデータ確認
-- POSTED_ATはTIMESTAMP型（setup.sqlで変換済み）
-- ============================================================================
SELECT 
    POST_ID,
    PLATFORM,
    USERNAME,
    SENTIMENT,
    LIKES,
    POSTED_AT,
    LEFT(CONTENT, 100) AS content_preview
FROM GOLD_SNS_MENTIONS_ANALYZED
LIMIT 5;

## 2. Cortex Search Serviceの作成

各データソースに対してCortex Search Serviceを作成します。

**共通パラメータ:**
- 埋め込みモデル: snowflake-arctic-embed-l-v2.0
- ターゲットラグ: 1日
- ウェアハウス: MCP_HANDSON_WH

### 2-1. FAQ用 Cortex Search Service

よくある質問（FAQ）を検索するサービスを作成します。

**設定項目:**
- サービス名: SEARCH_FAQ
- 対象テーブル: GOLD_FAQ_DOCUMENTS
- 検索列: CONTENT_CHUNK

In [ ]:
-- ============================================================================
-- FAQ用 Cortex Search Service の作成
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_FAQ
  ON CONTENT_CHUNK
  WAREHOUSE = MCP_HANDSON_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        RELATIVE_PATH,
        CONTENT_CHUNK
    FROM GOLD_FAQ_DOCUMENTS
    WHERE CONTENT_CHUNK IS NOT NULL
      AND LENGTH(TRIM(CONTENT_CHUNK)) > 0
  );

### 2-2. 運営マニュアル用 Cortex Search Service

業務マニュアルの検索サービスを作成します。

**設定項目:**
- サービス名: SEARCH_OPERATION_MANUALS
- 対象テーブル: GOLD_OPERATION_MANUALS
- 検索列: CONTENT_CHUNK

In [ ]:
-- ============================================================================
-- 運営マニュアル用 Cortex Search Service の作成
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_OPERATION_MANUALS
  ON CONTENT_CHUNK
  WAREHOUSE = MCP_HANDSON_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        RELATIVE_PATH,
        CONTENT_CHUNK
    FROM GOLD_OPERATION_MANUALS
    WHERE CONTENT_CHUNK IS NOT NULL
      AND LENGTH(TRIM(CONTENT_CHUNK)) > 0
  );

### 2-3. 音声ログ要約用 Cortex Search Service

コールセンターの通話要約を検索するサービスを作成します。

**設定項目:**
- サービス名: SEARCH_VOICE_LOGS
- 対象テーブル: GOLD_VOICE_LOGS
- 検索列: TRANSCRIBED_TEXT_SUMMARY
- 属性列: CALL_ID, CATEGORY, INQUIRY_CATEGORY, SENTIMENT

> **属性列（ATTRIBUTES）とは？**  
> 検索結果をフィルタリングするためのカラムです。  
> 例: 感情が「negative」の問い合わせのみを検索

In [ ]:
-- ============================================================================
-- 音声ログ要約用 Cortex Search Service の作成
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_VOICE_LOGS
  ON TRANSCRIBED_TEXT_SUMMARY
  ATTRIBUTES CALL_ID, CATEGORY, INQUIRY_CATEGORY, SENTIMENT
  WAREHOUSE = MCP_HANDSON_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        CALL_ID,
        SCENARIO_ID,
        CATEGORY,
        INQUIRY_CATEGORY,
        SENTIMENT,
        AGENT_ID,
        CALL_DURATION_SEC,
        CALL_START_TIME,
        TRANSCRIBED_TEXT_SUMMARY
    FROM GOLD_VOICE_LOGS
    WHERE TRANSCRIBED_TEXT_SUMMARY IS NOT NULL
      AND LENGTH(TRIM(TRANSCRIBED_TEXT_SUMMARY)) > 0
  );

### 2-4. SNS投稿分析用 Cortex Search Service

SNS上の顧客の声を検索するサービスを作成します。

**設定項目:**
- サービス名: SEARCH_SNS_MENTIONS
- 対象テーブル: GOLD_SNS_MENTIONS_ANALYZED
- 検索列: CONTENT
- 属性列: PLATFORM, POST_CATEGORY, SENTIMENT

In [ ]:
-- ============================================================================
-- SNS投稿分析用 Cortex Search Service の作成
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_SNS_MENTIONS
  ON CONTENT
  ATTRIBUTES PLATFORM, POST_CATEGORY, SENTIMENT
  WAREHOUSE = MCP_HANDSON_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        POST_ID,
        PLATFORM,
        POST_TYPE,
        USERNAME,
        DISPLAY_NAME,
        CONTENT,
        POSTED_AT,
        LIKES,
        RETWEETS,
        REPLIES,
        POST_CATEGORY,
        SENTIMENT,
        EXTRACTED_CATEGORY,
        EXTRACTED_PRODUCT_NAME
    FROM GOLD_SNS_MENTIONS_ANALYZED
    WHERE CONTENT IS NOT NULL
      AND LENGTH(TRIM(CONTENT)) > 0
  );

## 3. Cortex Search Service の確認

作成したCortex Search Serviceの一覧と状態を確認します。

In [ ]:
-- ============================================================================
-- Cortex Search Serviceの一覧確認
-- ============================================================================
SHOW CORTEX SEARCH SERVICES IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

## 4. 検索テスト

作成したCortex Search Serviceを使用して、基本的な検索を行います。

In [ ]:
-- ============================================================================
-- FAQ検索テスト: 返品に関するFAQ
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:RELATIVE_PATH::STRING AS relative_path,
    LEFT(f.value:CONTENT_CHUNK::STRING, 300) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_FAQ',
        '{
            "query": "返品の条件を教えてください",
            "columns": ["RELATIVE_PATH", "CONTENT_CHUNK"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- 運営マニュアル検索テスト: 返品対応手順
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:RELATIVE_PATH::STRING AS relative_path,
    LEFT(f.value:CONTENT_CHUNK::STRING, 300) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_OPERATION_MANUALS',
        '{
            "query": "返品の手続き方法",
            "columns": ["RELATIVE_PATH", "CONTENT_CHUNK"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- 音声ログ検索テスト: 配送遅延に関する問い合わせ
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:CALL_ID::STRING AS call_id,
    f.value:CATEGORY::STRING AS category,
    f.value:SENTIMENT::STRING AS sentiment,
    LEFT(f.value:TRANSCRIBED_TEXT_SUMMARY::STRING, 200) AS summary_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_VOICE_LOGS',
        '{
            "query": "配送が遅れている",
            "columns": ["CALL_ID", "CATEGORY", "SENTIMENT", "TRANSCRIBED_TEXT_SUMMARY"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- SNS投稿検索テスト: インテリアに関する口コミ
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    f.value:LIKES::INTEGER AS likes,
    TO_VARCHAR(f.value:POSTED_AT::TIMESTAMP_NTZ, 'YYYY-MM-DD HH24:MI') AS posted_at,
    LEFT(f.value:CONTENT::STRING, 100) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "インテリア おすすめ",
            "columns": ["POST_ID", "PLATFORM", "LIKES", "POSTED_AT", "CONTENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- 属性フィルター検索: ネガティブな問い合わせのみ
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:CALL_ID::STRING AS call_id,
    f.value:CATEGORY::STRING AS category,
    f.value:SENTIMENT::STRING AS sentiment,
    LEFT(f.value:TRANSCRIBED_TEXT_SUMMARY::STRING, 150) AS summary_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_VOICE_LOGS',
        '{
            "query": "商品の不具合",
            "columns": ["CALL_ID", "CATEGORY", "SENTIMENT", "TRANSCRIBED_TEXT_SUMMARY"],
            "filter": {"@eq": {"SENTIMENT": "negative"}},
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## まとめ

このノートブックでは、非構造化データに対するCortex Search Serviceを作成しました。

### 作成したCortex Search Service

| サービス名 | 用途 |
|-----------|------|
| SEARCH_FAQ | よくある質問への回答検索 |
| SEARCH_OPERATION_MANUALS | 業務手順・対応方法の検索 |
| SEARCH_VOICE_LOGS | 過去の顧客対応事例の検索 |
| SEARCH_SNS_MENTIONS | 顧客の声（VoC）の検索 |

### 次のステップ

- **Part 2**: Cortex Agent、MCP Server、PATの作成
  - Semantic Viewの作成
  - Cortex Agentの作成（Cortex Analyst + Cortex Search）
  - Snowflake Managed MCP Serverの作成
  - Programmatic Access Token（PAT）の発行
  - MCPクライアントからの接続テスト